# Task 3 e 4 — Prompt tuning shallow e deep nell'encoder di SAM (Kaggle, GPU)

Prompt tuning nella forma **prefix**: in un blocco dell'encoder, L chiavi e L valori addestrabili si aggiungono a quelli della griglia, con una softmax unica (`src/sampeft/prefix.py`).
- **Shallow (Task 3):** prefix solo nel blocco 0 → 12,288 parametri con L = 8.
- **Deep (Task 4):** prefix in tutti i 12 blocchi → 147,456 parametri, **gli stessi di LoRA r = 4**.

In entrambi si addestra anche il mask decoder, partendo da SAM originale. **Protocollo comune** della Task 5: 10 epoche, AMP, selezione sul Dice di validation, valutazione finale in float32.

Riferimenti (Task 5, test, box esatti): solo decoder **0.954 ± 0.003** · LoRA r = 4 **0.965 ± 0.003**.

**Session options:** *Accelerator* **GPU T4 x2** · *Internet* **On** · *Persistence* **Files only**.
Durata stimata: ~2.5 ore (due run da ~70 minuti, test, valutazioni). Consigliato: chiudi la sessione interattiva e usa *Save Version → Save & Run All*.

Ogni run, se ha già i risultati su disco, viene saltato. **L'ultima cella crea un link di download dei risultati leggeri** (metriche e CSV, senza pesi): i dati restano anche dentro il notebook salvato.

## 1. Codice della repo

In [ ]:
from pathlib import Path
import sys

REPO_URL = "https://github.com/caMatt99/sam-peft-kvasir.git"
WORK = Path("/kaggle/working")
REPO = WORK / "sam-peft-kvasir"

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}
!pip install -q -e "{REPO}[dev]"
%cd {REPO}
sys.path.insert(0, str(REPO / "src"))   # il kernel già avviato non vede l'install editable
!git log --oneline -1

## 2. Dati e pesi

In [ ]:
import torch
assert torch.cuda.is_available(), "Accendi la GPU in Session options → Accelerator"
print(torch.cuda.get_device_name(0))

def find_kvasir_root():
    for base in [Path("/kaggle/input"), WORK / "data"]:
        if base.exists():
            for p in base.rglob("images"):
                if (p.parent / "masks").is_dir():
                    return p.parent
    return None

RAW = find_kvasir_root()
if RAW is None:
    (WORK / "data").mkdir(parents=True, exist_ok=True)
    !wget -q -L --no-check-certificate https://datasets.simula.no/downloads/kvasir-seg.zip -O /kaggle/working/data/kvasir-seg.zip
    !unzip -q /kaggle/working/data/kvasir-seg.zip -d /kaggle/working/data/ && rm /kaggle/working/data/kvasir-seg.zip
    RAW = find_kvasir_root()
assert RAW is not None

PROCESSED = WORK / "processed"
if not (PROCESSED / "meta.json").exists():
    !python scripts/prepare_data.py --raw-root {RAW} --out {PROCESSED} > /dev/null

CKPT = WORK / "checkpoints" / "sam_vit_b_01ec64.pth"
if not CKPT.exists() or CKPT.stat().st_size < 300e6:
    CKPT.parent.mkdir(parents=True, exist_ok=True)
    !wget -q -L https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {CKPT}

CACHE = WORK / "cache" / "vit_b"
COMMON = f"--raw-root {RAW} --processed {PROCESSED} --checkpoint {CKPT}"
done = lambda run: Path(f"{run}/test/metrics.json").exists()
print(RAW, "|", PROCESSED, "| checkpoint MB:", round(CKPT.stat().st_size / 1e6))

## 3. Test
Nuovi in questa task (`tests/test_prefix.py`): con 0 prefix l'attenzione coincide esattamente con quella di SAM (blocchi a finestre e globali), conta dei parametri, gradienti solo sui prefix, salvataggio e caricamento, training da script, export dei risultati.

In [ ]:
!python -m pytest -q

## 4. Protocollo comune
Fissato nella Task 5 dopo il benchmark. Non va cambiato: è ciò che rende i metodi confrontabili.

In [ ]:
EPOCHS = 10
AMP = True
LENGTH = 8

FLAGS = f"--epochs {EPOCHS}" + (" --amp" if AMP else "")
RUNS = {"prefix shallow": "runs/prefix_shallow", "prefix deep": "runs/prefix_deep"}
REF = {"solo decoder (Task 5)": (0.954, 0.003), "LoRA r=4 (Task 5)": (0.965, 0.003)}
print(FLAGS); print(RUNS)

## 5. Zero-shot sul test
Righe di riferimento e controllo dei tempi di inferenza sulla stessa GPU (≈2 minuti).

In [ ]:
if not done("runs/zeroshot"):
    !python scripts/evaluate.py {COMMON} --split test --out runs/zeroshot

## 6. Training
**Controllo dell'epoca 0:** con la softmax unica i prefix "rubano" un po' di attenzione già prima del training, quindi l'epoca 0 non sarà esattamente 0.906 come per LoRA. Quanto se ne discosta misura il disturbo iniziale: va annotato.
`*` = nuova migliore epoca sulla validation.

In [ ]:
for method, run in [("prefix_shallow", RUNS["prefix shallow"]), ("prefix_deep", RUNS["prefix deep"])]:
    if done(run):
        print("già fatto:", run); continue
    print(f"\n========== {method} (L = {LENGTH}) ==========")
    !python scripts/train.py {COMMON} --method {method} --length {LENGTH} {FLAGS} --out {run}
    for s in ["val", "test"]:
        !python scripts/evaluate.py {COMMON} --split {s} --weights {run}/trainable.pt --out {run}

## 7. Analisi
### Tabella (test)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

load = lambda run, split_name="test": json.loads(Path(f"{run}/{split_name}/metrics.json").read_text())
summ = {name: json.loads(Path(f"{run}/summary.json").read_text()) for name, run in RUNS.items()}
fmt = lambda g, k: f"{g[f'{k}.dice_raw']['mean']:.3f} ± {g[f'{k}.dice_raw']['sem']:.3f}"

rows = []
for name, run in {"zero-shot": "runs/zeroshot", **RUNS}.items():
    m = load(run); g = m["groups"]["all"]; s = summ.get(name)
    rows.append({
        "metodo": name,
        "param. encoder": s["parameters"]["encoder"] if s else 0,
        "param. addestrati": s["parameters"]["trainable"] if s else 0,
        "Dice": fmt(g, "boxes"), "Dice jitter": fmt(g, "boxes_jitter"),
        "fail": f"{g['boxes.fail_rate']:.0%}",
        "large_polyp": f"{m['groups']['large_polyp']['boxes.dice_raw']['mean']:.3f}" if "large_polyp" in m["groups"] else "-",
        "s / epoca": round(s["train_s_per_epoch_median"]) if s else None,
        "training (min)": round(s["train_s_total"] / 60, 1) if s else None,
        "memoria (GB)": round(s["peak_gpu_memory_mb"] / 1024, 1) if s and s["peak_gpu_memory_mb"] else None,
        "epoca 0 (val)": round(json.loads(Path(f"{run}/history.json").read_text())[0]["val_dice"], 3) if s else None,
        "epoca migliore": s["best_epoch"] if s else None,
        "inferenza encoder (ms)": round(m["timing_median_ms"]["encoder_ms"]),
    })
display(pd.DataFrame(rows).set_index("metodo"))
print("Riferimenti Task 5 (test, box esatti):", REF)

### Confronto appaiato shallow − zero-shot e deep − shallow (test)
Il confronto appaiato con decoder e LoRA richiede i risultati per immagine della Task 5: lo facciamo fuori dal notebook (o nella Task 6) con i due zip.

In [ ]:
per = lambda run: pd.read_csv(f"{run}/test/per_image.csv").set_index("id")
zs, sh, dp = per("runs/zeroshot"), per(RUNS["prefix shallow"]), per(RUNS["prefix deep"])
for label, a, b in [("shallow − zero-shot", sh, zs), ("deep − zero-shot", dp, zs), ("deep − shallow", dp, sh)]:
    for k in ["boxes", "boxes_jitter"]:
        diff = a.loc[zs.index, f"{k}.dice_raw"] - b.loc[zs.index, f"{k}.dice_raw"]
        print(f"{label:<20} {k:<13} {diff.mean():+.4f} ± {diff.std(ddof=1) / np.sqrt(len(diff)):.4f}"
              f"   migliora {(diff > 0.01).sum():3d} · peggiora {(diff < -0.01).sum():3d}")

### Curve di training

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
for name, run in RUNS.items():
    h = pd.DataFrame(json.loads(Path(f"{run}/history.json").read_text()))
    ax[0].plot(h["epoch"][1:], h["train_loss"][1:], label=name)
    ax[1].plot(h["epoch"], h["val_dice"], label=name)
    ax[2].plot(h["epoch"], h["val_dice_jitter"], label=name)
for a, t in zip(ax, ["loss di training", "val Dice, box esatti", "val Dice, box con jitter"]):
    a.set_title(t); a.set_xlabel("epoca"); a.legend()
ax[1].axhline(0.906, ls=":", c="gray"); ax[2].axhline(0.882, ls=":", c="gray")   # zero-shot su val
plt.show()

### Per dimensione del polipo (test, box esatti)

In [ ]:
tab = pd.DataFrame({"area": zs["area_frac"], "zero-shot": zs["boxes.dice_raw"],
                    **{n: per(r).loc[zs.index, "boxes.dice_raw"] for n, r in RUNS.items()}})
tab["bin"] = pd.cut(tab["area"], [0, .1, .25, .5, 1], labels=["<10%", "10-25%", "25-50%", ">50%"])
display(tab.groupby("bin", observed=True).agg(n=("area", "size"),
        **{c: (c, "mean") for c in ["zero-shot", *RUNS]}).round(3))

## 8. Download dei risultati
- **Link qui sotto:** zip leggero (metriche, CSV per immagine, curve, riassunti; senza i pesi). Un clic e si scarica. Il contenuto è salvato anche nel notebook: se mi mandi il .ipynb, i risultati li estraggo da lì.
- Zip completo, con i pesi addestrati: `/kaggle/working/task34_results.zip` nel pannello Output (serve solo se vuoi i pesi).

In [ ]:
from IPython.display import HTML, display
from sampeft.export import download_link_html

display(HTML(download_link_html("runs", "task34_results_light.zip")))
!rm -f {WORK}/task34_results.zip
!zip -qr {WORK}/task34_results.zip runs/ -x "runs/zeroshot/*"
!ls -lh {WORK}/task34_results.zip

## Da annotare
1. Epoca 0: quanto si discosta da 0.906 per shallow e deep?
2. Dice sul test, box esatti e jitter: dove si collocano rispetto a decoder (0.954) e LoRA r = 4 (0.965)?
3. Deep − shallow, confronto appaiato: i prefix in tutti i blocchi servono?
4. Costo: s/epoca, memoria, inferenza (i prefix restano in inferenza: quanto costano rispetto a SAM?).